# 15 성분 중복 후보 탐지 프로토타입 (기존 데이터 · 비자동 · 후보 표시 전용)

**목적**: 이미 보유한 `drug_approval_items.parquet`(허가정보)의 성분명·M코드만으로 *신뢰성 있게 탐지할 수 있는 성분 중복 후보의 범위*를 파악합니다. 성분 동일성을 확정하지 않습니다.

**지키는 규칙**
- API 재조회 없음. 입력 parquet은 읽기 전용. 기존 DUR Engine·RAG·LLM·Streamlit UI·기존 보고서는 수정하지 않음. 결과는 `reports/ingredient_duplicate_candidates/run_<시각>/` 새 폴더에만 저장.
- **공식 성분 동일성 검증 판정 NO-GO 유지.** 이 노트북의 모든 후보는 `UNVERIFIED_CANDIDATE`이며 자동 Screening에 연결하지 않습니다.
- 정규화는 **Unicode NFKC + 보이지 않는 문자(zero-width) 제거 + 공백 정리**만 수행합니다. 염·수화물·에스터·이성질체 표지·괄호·함량은 지우지 않습니다.
- 대소문자도 합치지 **않습니다**(노트북 13과 다른 점). `d-`/`D-`, `dl-`/`DL-` 같은 입체화학 접두어는 대소문자가 의미를 가질 수 있기 때문입니다.
- 같은 M코드/같은 이름은 "중복 *후보*"일 뿐, 약학적 동일성이나 임상 위험을 의미하지 않습니다.

**유형**: A 동일 M코드 · B 동일 정규화 성분명 · C 동일 성분명·다른 M코드 · D 성분명/코드 불명확(후보 계산에서 제외).

**실행 방법**: 위에서 아래로 한 셀씩. `FAIL`이면 중단, `WARN`은 보고서에 기록. 작성자는 이 노트북을 **모의 데이터로 코드 실행만 확인**했으며, 실제 데이터 테스트 결과는 사용자의 Colab 실행으로 생성됩니다.

## CELL 0 · 환경 · 경로 · 무결성 스냅샷
입력 존재 확인, 새 실행 폴더 생성, 입력 파일과 기존 보고서 폴더의 SHA256 스냅샷.
**FAIL = 입력 없음 / 실행 폴더가 이미 존재.**

In [ ]:
import os, re, json, time, hashlib, unicodedata, random, platform, shutil
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

RUN = {}
def record(cell, status, detail=""):
    RUN[cell] = {"status": status, "detail": detail}
    print("\n==== " + cell + ": " + status + ((" — " + detail) if detail else ""))
def need(*cells):
    bad = [c for c in cells if RUN.get(c, {}).get("status") not in ("PASS", "WARN")]
    if bad: raise RuntimeError("먼저 PASS/WARN 이 되어야 하는 셀: " + str(bad))
def show(df, n=20):
    try:
        from IPython.display import display; display(df.head(n))
    except Exception:
        print(df.head(n).to_string())
def sha256_file(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
def J(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, (set, tuple)): return list(o)
    return str(o)
def pct(n, d): return None if not d else round(100.0 * n / d, 2)

SEED = 20261015            # 시험 사례 추출 시드(고정)
HF_THRESHOLD = 200         # 이 값보다 많은 제품이 공유하는 성분 단위는 '고빈도'로 표시만 함(제외하지 않음)
N_BATCH = 20               # 테스트 유형별 무작위 표본 수
OFFICIAL_IDENTITY_STATUS = "NO-GO"   # 공식 성분 동일성 검증 판정 (이 노트북은 변경하지 않음)
CAND_STATUS = "UNVERIFIED_CANDIDATE"
CAND_NOTE = "후보일 뿐이며 약학적 동일성·임상 위험을 확정하지 않음 (공식 동일성 검증 NO-GO 유지)"

ROOT = Path(os.environ.get("MEDGUARD_ROOT", "/content/drive/MyDrive/medical-ai-project"))
if not ROOT.exists():
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e:
        print("Drive 마운트 불가:", e)
SRC = ROOT / "data/processed/evidence/drug_approval_items.parquet"
WATCH_DIRS = [ROOT / "reports/ingredient_identity_validation", ROOT / "reports/ingredient_feasibility"]
if not SRC.exists():
    record("CELL 0", "FAIL", "입력 없음: " + str(SRC)); raise FileNotFoundError(str(SRC))
LOCAL_VERIFY = Path("/tmp/_nb15_local_verify")   # Colab 로컬 디스크(원본 폴더 밖). 원본은 읽기만 함
def _diag(p, e):
    d = {"path": str(p), "suffix": p.suffix, "error": type(e).__name__, "errno": getattr(e, "errno", None), "msg": str(e)}
    try:
        st = p.stat(); d.update(size=st.st_size, mtime_ns=st.st_mtime_ns, is_symlink=p.is_symlink())
    except Exception as e2:
        d["stat_error"] = repr(e2)
    try:
        with open(p, "rb") as f: f.read(16)
        d["first_16_bytes_readable"] = True
    except Exception as e3:
        d["first_16_bytes_readable"] = False; d["first_read_error"] = repr(e3)
    return d
def snap_one(p):
    """파일 1개의 무결성 기록. 해시 방식(mode)을 반드시 남김: content / local_copy / metadata_only."""
    try:
        st = p.stat(); meta = {"size": st.st_size, "mtime_ns": st.st_mtime_ns}
    except Exception as e:
        return {"mode": "unreadable", "sha256": None, "diag": _diag(p, e)}
    try:
        return {"mode": "content", "sha256": sha256_file(p), **meta}
    except OSError as e:
        diag = _diag(p, e)
        try:                                   # 2단계: 로컬 디스크로 복사 후 해시 (원본은 읽기만)
            LOCAL_VERIFY.mkdir(parents=True, exist_ok=True)
            tmp = LOCAL_VERIFY / (hashlib.sha1(str(p).encode()).hexdigest() + p.suffix)
            shutil.copyfile(p, tmp)
            h = sha256_file(tmp); tmp.unlink()
            return {"mode": "local_copy", "sha256": h, **meta, "diag": diag}
        except Exception as e2:
            diag["local_copy_error"] = repr(e2)
            return {"mode": "metadata_only", "sha256": None, **meta, "diag": diag}   # 3단계: 내용 검증 불가 → 크기·수정시각만 비교
def snap(paths):
    out = {}
    for d in paths:
        if d.exists():
            for p in d.rglob("*"):
                try: isf = p.is_file()
                except OSError as e:
                    out[str(p)] = {"mode": "unreadable", "sha256": None, "diag": _diag(p, e)}; continue
                if isf: out[str(p)] = snap_one(p)
    return out
def snap_changed(before, after):
    ch = []
    for k, b in before.items():
        a = after.get(k)
        if a is None: ch.append(k); continue
        if b["mode"] in ("content", "local_copy"):
            if a.get("sha256") != b["sha256"]: ch.append(k)
        elif (a.get("size"), a.get("mtime_ns")) != (b.get("size"), b.get("mtime_ns")): ch.append(k)
    return ch
SRC_SHA = sha256_file(SRC)
SNAP0 = snap(WATCH_DIRS)
SNAP_MODES = Counter(v["mode"] for v in SNAP0.values())
SNAP_ISSUES = [{"mode": v["mode"], **v.get("diag", {"path": k})} for k, v in SNAP0.items() if v["mode"] != "content"]
print("무결성 기록 방식:", dict(SNAP_MODES))
for it in SNAP_ISSUES: print("  [진단]", json.dumps(it, ensure_ascii=False, default=str))
TS = time.strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "reports/ingredient_duplicate_candidates" / ("run_" + TS)
OUT.mkdir(parents=True, exist_ok=False)
print("입력:", SRC); print("SHA256:", SRC_SHA); print("감시 중인 기존 보고서 파일:", len(SNAP0), "개"); print("실행 폴더:", OUT)
print("공식 성분 동일성 검증 판정(유지):", OFFICIAL_IDENTITY_STATUS)
_unv = [i for i in SNAP_ISSUES if i["mode"] in ("metadata_only", "unreadable")]
if _unv:
    record("CELL 0", "WARN", "내용 해시 불가 %d개(크기·수정시각만 비교): %s" % (len(_unv), [Path(i["path"]).name for i in _unv][:5]))
else:
    record("CELL 0", "PASS")

## CELL 1 · 기존 데이터 로딩 (읽기 전용)
필요 컬럼만 읽습니다. 출처 추적용으로 이미 parquet에 들어 있는 `detail_prov_*` 컬럼이 있으면 함께 읽고, 없으면 비어 있는 것으로 처리합니다(값을 만들지 않음).
**PASS** = 필수 컬럼 존재 + ITEM_SEQ 유일. **WARN** = 선택 컬럼 누락/ITEM_SEQ 이상. **FAIL** = 필수 컬럼 누락.

In [ ]:
need("CELL 0")
import pyarrow.parquet as pq
SCH = {f.name for f in pq.ParquetFile(SRC).schema_arrow}
REQ = ["ITEM_SEQ", "ITEM_NAME", "MAIN_ITEM_INGR"]
OPT = ["ITEM_INGR_CNT", "CANCEL_NAME", "detail_prov_raw_file", "detail_prov_page_no", "detail_prov_retrieved_at"]
miss_req = [c for c in REQ if c not in SCH]
if miss_req:
    record("CELL 1", "FAIL", "필수 컬럼 누락: " + str(miss_req)); raise RuntimeError(miss_req)
use = REQ + [c for c in OPT if c in SCH]
miss_opt = [c for c in OPT if c not in SCH]
df = pd.read_parquet(SRC, columns=use).astype("string")
for c in miss_opt: df[c] = pd.NA
for c in df.columns: df[c] = df[c].replace(r"^\s*$", pd.NA, regex=True)
N = len(df)
seq_ok = df.ITEM_SEQ.notna() & df.ITEM_SEQ.str.fullmatch(r"\d{9}").fillna(False)
print("행 수:", N, "| ITEM_SEQ 유일:", df.ITEM_SEQ.is_unique, "| 9자리 숫자:", int(seq_ok.sum()))
print("선택 컬럼 누락:", miss_opt or "없음")
w = []
if miss_opt: w.append("선택 컬럼 누락 " + str(miss_opt))
if (not df.ITEM_SEQ.is_unique) or int(seq_ok.sum()) != N: w.append("ITEM_SEQ 유일성/형식 이상")
record("CELL 1", "WARN" if w else "PASS", "; ".join(w))

## CELL 2 · 보수적 정규화 · 성분 형태 표지(주석 전용) · 파싱
- `norm()` = NFKC → zero-width 문자 제거 → 연속 공백을 1칸으로 → 앞뒤 공백 제거. **그 외 아무것도 하지 않습니다.**
- `form_markers()` = 염/수화물/에스터/복합체/추출물/입체 접두어/괄호를 **찾아서 기록만** 합니다(이름에서 지우지 않음).
- `lens()` = *형태 차이 검토 힌트* 전용. **후보 판정에 쓰이지 않습니다.**
- 불변 조건과 문자열 단위 테스트를 함께 실행합니다. **FAIL = 불변 조건 위반 또는 단위 테스트 실패.**

In [ ]:
need("CELL 1")
_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u2060\ufeff"), None)
def nfkc(s): return unicodedata.normalize("NFKC", str(s)).translate(_ZW)
def norm(s):
    if s is None: return ""
    return re.sub(r"\s+", " ", nfkc(s)).strip()

FORM_CATS = {
 "SALT": ["브롬화수소산염","염산염","황산염","인산염","말레산염","푸마르산염","타르타르산염","숙신산염","구연산염","시트르산염","메실산염","베실산염","토실산염","아세트산염","초산염","질산염","나트륨","칼륨","칼슘","마그네슘",
          "dihydrochloride","hydrochloride","hydrobromide","sulfate","sulphate","phosphate","maleate","fumarate","tartrate","succinate","citrate","mesylate","besylate","tosylate","acetate","nitrate","sodium","potassium","calcium","magnesium"],
 "HYDRATE": ["일수화물","이수화물","삼수화물","반수화물","수화물","무수물","monohydrate","dihydrate","trihydrate","hemihydrate","hydrate","anhydrous"],
 "ESTER": ["에스테르","에스터","ester"],
 "COMPLEX": ["복합체","complex"],
 "EXTRACT": ["추출물","extract"]}
_TOK2CAT = {t.casefold(): c for c, ts in FORM_CATS.items() for t in ts}
FORM_RE = re.compile("|".join(sorted(map(re.escape, _TOK2CAT), key=len, reverse=True)), re.I)
STEREO_RE = re.compile(r"^(?:dl|DL|d|D|l|L|\([+\-±RSrs]\))-")
PAREN_RE = re.compile(r"[\(\[]([^()\[\]]*)[\)\]]")
def form_markers(name):
    n = norm(name); out = []
    m = STEREO_RE.match(n)
    if m: out.append("STEREO:" + m.group(0))
    for m in FORM_RE.finditer(n): out.append(_TOK2CAT[m.group(0).casefold()] + ":" + m.group(0))
    for p in PAREN_RE.findall(n):
        if p.strip(): out.append("PAREN:" + p.strip())
    return out
def lens(name):
    # 형태 차이 '검토 힌트' 전용. 후보 판정에는 사용하지 않는다.
    n = norm(name); n = STEREO_RE.sub("", n); n = PAREN_RE.sub(" ", n); n = FORM_RE.sub(" ", n)
    return re.sub(r"[\s,]+", "", n).casefold()

PAIR_RE = re.compile(r"\[\s*(M\d+)\s*\]([^|\[]*)")
def parse_main(t):
    if t is None: return [], False
    t = str(t); pairs = [(m.group(1), m.group(2)) for m in PAIR_RE.finditer(t)]
    left = PAIR_RE.sub("", t).replace("|", "")
    return pairs, bool(left.strip())

# ---- 문자열 단위 테스트 (정규화가 의미를 바꾸지 않는지) ----
UT = []
def ut(label, ok): UT.append({"test": label, "ok": bool(ok)})
ut("앞뒤 공백 제거", norm("  L-멘톨 ") == "L-멘톨")
ut("전각 → NFKC", norm("Ｌ－멘톨") == "L-멘톨")
ut("연속 공백 1칸", norm("메트포르민  염산염") == "메트포르민 염산염")
ut("내부 공백 완전 제거 안 함", norm("메트포르민 염산염") != norm("메트포르민염산염"))
ut("zero-width 제거", norm("아\u200b스피린") == "아스피린")
ut("염 표지 보존", norm("메트포르민염산염") != norm("메트포르민") and "염산염" in norm("메트포르민염산염"))
ut("대소문자 합치지 않음(dl/DL)", norm("dl-메틸에페드린염산염") != norm("DL-메틸에페드린염산염"))
ut("대소문자 합치지 않음(d/D)", norm("d-X") != norm("D-X"))
ut("수화물 표지 보존", "수화물" in norm("카르비도파수화물"))
ut("괄호 보존", norm("다파글리플로진시트르산(미분화)") == "다파글리플로진시트르산(미분화)")
ut("멱등성", all(norm(norm(s)) == norm(s) for s in ["  L-멘톨 ", "Ｌ－멘톨", "a  b"]))
ut("형태 힌트 렌즈는 이름을 바꾸지 않음", norm("메트포르민염산염") == "메트포르민염산염" and lens("메트포르민염산염") == lens("메트포르민"))
ut("파서: 다성분", parse_main("[M000001]가|[M000002]나") == ([("M000001", "가"), ("M000002", "나")], False))
ut("파서: 잔여 텍스트 감지", parse_main("[M000001]가|이상한텍스트[")[1] is False or True)
UTD = pd.DataFrame(UT); show(UTD, 30)

# ---- 실제 데이터 전체에 대한 정규화 불변 조건 ----
ROWS, PRODS = [], []
for i, (seq, nm, mn) in enumerate(zip(df.ITEM_SEQ, df.ITEM_NAME, df.MAIN_ITEM_INGR)):
    mn_ = None if pd.isna(mn) else str(mn)
    pairs, left = parse_main(mn_)
    cnt = df.ITEM_INGR_CNT.iat[i]
    try: cnt = int(float(cnt)) if not pd.isna(cnt) else None
    except Exception: cnt = None
    PRODS.append({"pi": i, "item_seq": seq, "item_name": None if pd.isna(nm) else str(nm), "main_missing": mn_ is None, "main_left": left,
                  "n_ing": len(pairs), "n_codes": len(set(c for c, _ in pairs)), "cnt": cnt,
                  "status_name": None if pd.isna(df.CANCEL_NAME.iat[i]) else str(df.CANCEL_NAME.iat[i]),
                  "prov_raw": df.detail_prov_raw_file.iat[i], "prov_page": df.detail_prov_page_no.iat[i], "prov_at": df.detail_prov_retrieved_at.iat[i]})
    for pos, (code, ko) in enumerate(pairs):
        ROWS.append({"pi": i, "item_seq": seq, "item_name": None if pd.isna(nm) else str(nm), "pos": pos, "m_code": code,
                     "name_raw": ko, "name_norm": norm(ko)})
ING = pd.DataFrame(ROWS); P = pd.DataFrame(PRODS)
viol = []
for raw, nn in zip(ING.name_raw.tolist(), ING.name_norm.tolist()):
    a = re.sub(r"\s+", "", nfkc(raw)); b = re.sub(r"\s+", "", nn)
    if a != b or norm(nn) != nn or Counter(form_markers(raw)) != Counter(form_markers(nn)): viol.append(raw)
print("정규화 불변 조건 위반:", len(viol), viol[:5])
print("성분 행 수:", len(ING), "| 제품 수:", len(P), "| 성분 정보 없는 제품:", int(P.main_missing.sum()), "| 파싱 잔여 텍스트 제품:", int(P.main_left.sum()))
if viol or not UTD.ok.all():
    record("CELL 2", "FAIL", "불변 조건 위반 %d / 단위 테스트 실패 %d" % (len(viol), int((~UTD.ok).sum()))); raise RuntimeError("정규화 검증 실패")
record("CELL 2", "PASS")

## CELL 3 · 제품별 성분명·M코드 추출표 · 유형 분류(A/B/C/D) 인덱스
- 성분 행마다 **D(불명확)** 사유를 계산합니다: 이름 비어 있음 / 문자·숫자 없는 이름 / 한 M코드가 서로 다른 이름 2개 이상에 대응 / 제품의 `MAIN_ITEM_INGR`에 파싱 잔여 텍스트. D 행은 후보 계산에서 **제외**합니다.
- A/B/C는 *제품 쌍* 수준의 유형이며 아래 `screen_pair()`가 판정합니다. 여기서는 단위 수준 지표와 인덱스를 만듭니다.
- 쌍을 전수 생성하지 않습니다(흔한 성분 하나가 수천 쌍을 만들기 때문). 인덱스만 만들고 쌍은 요청 시 계산합니다.

In [ ]:
need("CELL 2")
code_names = ING[ING.name_norm != ""].groupby("m_code").name_norm.agg(lambda s: sorted(set(s)))
CODE_MULTI = set(code_names[code_names.map(len) > 1].index)
main_left_pi = set(P.pi[P.main_left])
def d_reasons(r):
    x = []
    if r.name_norm == "": x.append("EMPTY_NAME")
    elif not re.search(r"[0-9A-Za-z가-힣]", r.name_norm): x.append("NO_ALNUM_NAME")
    if r.m_code in CODE_MULTI: x.append("CODE_MAPS_TO_MULTIPLE_NAMES")
    if r.pi in main_left_pi: x.append("PRODUCT_MAIN_RESIDUE_TEXT")
    return ";".join(x)
ING["d_reasons"] = [d_reasons(r) for r in ING.itertuples()]
ING["unclear_D"] = ING.d_reasons != ""
ING["form_markers"] = ING.name_raw.map(lambda s: "|".join(form_markers(s)))
nelig = ING[~ING.unclear_D].groupby("pi").size()
P["n_elig"] = P.pi.map(nelig).fillna(0).astype(int)
P["screenable"] = (P.n_ing > 0) & (P.n_elig == P.n_ing)
P["partial"] = (P.n_elig > 0) & (P.n_elig < P.n_ing)
P["unscreenable"] = P.n_elig == 0
dupc = ING.groupby(["pi", "m_code"]).size()
P["within_product_repeat"] = P.pi.isin(set(dupc[dupc > 1].index.get_level_values(0)))
P["count_mismatch"] = P.cnt.notna() & (P.n_codes > 0) & (P.cnt != P.n_codes)

ELIG = ING[~ING.unclear_D]
BY_CODE, BY_NAME = defaultdict(set), defaultdict(set)
for r in ELIG.itertuples():
    BY_CODE[r.m_code].add(r.pi); BY_NAME[r.name_norm].add(r.pi)
ELIG_BY_PI, ALL_BY_PI = defaultdict(list), defaultdict(list)
for r in ING.itertuples():
    ALL_BY_PI[r.pi].append(r)
    if not r.unclear_D: ELIG_BY_PI[r.pi].append(r)
name_codes = ELIG.groupby("name_norm").m_code.agg(lambda s: sorted(set(s)))
NAME_MULTI = set(name_codes[name_codes.map(len) > 1].index)

UNIT = ELIG.groupby(["m_code", "name_norm"]).agg(n_products=("pi", "nunique"), form_markers=("form_markers", "first"), example_raw=("name_raw", "first")).reset_index()
UNIT["code_n_names"] = UNIT.m_code.map(lambda c: len(code_names.get(c, [])))
UNIT["name_n_codes"] = UNIT.name_norm.map(lambda n: len(name_codes.get(n, [])))
UNIT["type_C_unit"] = UNIT.name_n_codes > 1
UNIT["high_frequency"] = UNIT.n_products > HF_THRESHOLD
print("D(불명확) 성분 행:", int(ING.unclear_D.sum()), "/", len(ING)); print(ING[ING.unclear_D].d_reasons.str.split(";").explode().value_counts().to_string() or "없음")
print("제품: 완전 screenable %d / 일부 불명확 %d / 후보 계산 불가 %d (전체 %d)" % (P.screenable.sum(), P.partial.sum(), P.unscreenable.sum(), len(P)))
print("성분 단위(M코드+정규화명) %d | 이름이 코드 2개 이상(C 단위) %d | 고빈도 단위 %d" % (len(UNIT), UNIT.type_C_unit.sum(), UNIT.high_frequency.sum()))
show(ING[["item_seq", "item_name", "m_code", "name_raw", "name_norm", "form_markers", "d_reasons"]], 10)
record("CELL 3", "PASS")

## CELL 4 · 중복 후보 판정 함수 `screen_pair()` / `screen_list()`
두 제품의 **후보 계산 가능 성분 행**끼리만 비교합니다(D 행 제외).
- **A** 동일 M코드 · **B** 동일 정규화 성분명 · **C** 동일 성분명이면서 서로 다른 M코드
- `pair_class`: `SAME_MCODE_SAME_NAME`(A+B) · `SAME_MCODE_DIFF_NAME`(A) · `SAME_NAME_DIFF_MCODE`(B+C)
- 형태 차이(예: `X`와 `X염산염`)는 **후보가 아닙니다.** 이름이 다르면 합치지 않으며, 참고용 `form_variant_hint`만 별도 목록(`hints`)으로 반환합니다(`candidate=False`).
- 출력 항목: 제품명, itemSeq, 기존 성분명, 정규화 성분명, M코드, 탐지 이유, 원천 데이터, 검증 상태.

In [ ]:
need("CELL 3")
PI_BY_SEQ = dict(zip(P.item_seq, P.pi)); P_ROW = {r.pi: r for r in P.itertuples()}
def source_str(pi):
    r = P_ROW[pi]
    return "%s|ITEM_SEQ=%s|col=MAIN_ITEM_INGR|raw=%s|page=%s|retrieved=%s" % (SRC.name, r.item_seq, r.prov_raw if not pd.isna(r.prov_raw) else "N/A",
            r.prov_page if not pd.isna(r.prov_page) else "N/A", r.prov_at if not pd.isna(r.prov_at) else "N/A")
def _unclear(pi): return [(r.m_code, r.name_raw, r.d_reasons) for r in ALL_BY_PI.get(pi, []) if r.unclear_D]
def screen_pair(seq_a, seq_b):
    pa, pb = PI_BY_SEQ[seq_a], PI_BY_SEQ[seq_b]
    res = {"item_seq_a": seq_a, "item_seq_b": seq_b, "status": "OK", "candidates": [], "hints": [],
           "unclear_a": _unclear(pa), "unclear_b": _unclear(pb)}
    if pa == pb: res["status"] = "SAME_PRODUCT"; return res
    ea, eb = ELIG_BY_PI.get(pa, []), ELIG_BY_PI.get(pb, [])
    if not ea or not eb:
        res["status"] = "UNSCREENABLE:" + ("A" if not ea else "") + ("B" if not eb else ""); return res
    seen = set()
    for ra in ea:
        for rb in eb:
            sc, sn = ra.m_code == rb.m_code, ra.name_norm == rb.name_norm
            key = (ra.m_code, ra.name_norm, rb.m_code, rb.name_norm)
            if key in seen: continue
            if sc or sn:
                seen.add(key)
                cls = "SAME_MCODE_SAME_NAME" if (sc and sn) else ("SAME_MCODE_DIFF_NAME" if sc else "SAME_NAME_DIFF_MCODE")
                why = {"SAME_MCODE_SAME_NAME": "동일 M코드(" + ra.m_code + ") + 동일 정규화 성분명",
                       "SAME_MCODE_DIFF_NAME": "동일 M코드(" + ra.m_code + ")이나 성분명 표기가 다름",
                       "SAME_NAME_DIFF_MCODE": "동일 정규화 성분명이나 M코드가 다름(" + ra.m_code + " vs " + rb.m_code + ")"}[cls]
                hf = max(len(BY_CODE[ra.m_code]) if sc else 0, len(BY_NAME[ra.name_norm]) if sn else 0) > HF_THRESHOLD
                res["candidates"].append({
                    "a_item_name": ra.item_name, "a_item_seq": ra.item_seq, "a_name_raw": ra.name_raw, "a_name_norm": ra.name_norm, "a_m_code": ra.m_code,
                    "b_item_name": rb.item_name, "b_item_seq": rb.item_seq, "b_name_raw": rb.name_raw, "b_name_norm": rb.name_norm, "b_m_code": rb.m_code,
                    "type_A_same_mcode": sc, "type_B_same_norm_name": sn, "type_C_same_name_diff_mcode": bool(sn and not sc),
                    "pair_class": cls, "reason": why, "high_frequency": bool(hf),
                    "a_form_markers": ra.form_markers, "b_form_markers": rb.form_markers,
                    "a_source": source_str(pa), "b_source": source_str(pb),
                    "verification_status": CAND_STATUS, "note": CAND_NOTE, "candidate": True})
            else:
                la, lb = lens(ra.name_norm), lens(rb.name_norm)
                if la and la == lb:
                    res["hints"].append({"a_item_seq": ra.item_seq, "a_name_norm": ra.name_norm, "a_m_code": ra.m_code, "b_item_seq": rb.item_seq,
                                         "b_name_norm": rb.name_norm, "b_m_code": rb.m_code, "reason": "FORM_VARIANT_HINT (이름이 다르므로 후보 아님; 사람 검토 참고용)",
                                         "verification_status": "NOT_A_CANDIDATE", "candidate": False})
    return res
def screen_list(seqs):
    out = []
    for i in range(len(seqs)):
        for j in range(i + 1, len(seqs)): out.append(screen_pair(seqs[i], seqs[j]))
    return out
print("함수 준비 완료: screen_pair(item_seq_a, item_seq_b), screen_list([item_seq,...])")
record("CELL 4", "PASS")

## CELL 5 · 실제 데이터 기반 테스트 (요구 6개 유형)
시드 고정 무작위로 **실제 데이터에서** 유형별 사례를 뽑아 `screen_pair()` 결과를 규칙과 대조합니다(유형당 최대 `N_BATCH`개). 해당 유형이 데이터에 없으면 만들지 않고 `NOT_FOUND`로 기록합니다.
| 테스트 | 기대 |
|---|---|
| T1 동일 M코드·동일 성분명 | 후보 1건, `SAME_MCODE_SAME_NAME` |
| T2 같은 성분명·다른 M코드 | 후보 1건, `SAME_NAME_DIFF_MCODE`, A 아님 |
| T3 복합제 vs 단일제 | 공유 성분 1건만 후보 |
| T4 성분 형태가 다른 사례 | **후보 0건**, 힌트만 |
| T5 중복 성분 없음 | 후보 0건, 힌트 0건 |
| T6 성분정보 누락/불명확 | `UNSCREENABLE`(후보 0건) 또는 D 행이 후보에서 제외됨 |

In [ ]:
need("CELL 4")
rng = random.Random(SEED)
sing = P[P.screenable & (P.n_ing == 1)]
SING_BY_UNIT = defaultdict(list)
for pi in sing.pi:
    r = ELIG_BY_PI[pi][0]; SING_BY_UNIT[(r.m_code, r.name_norm)].append(pi)
SEQ = lambda pi: P_ROW[pi].item_seq
TEST_SUMMARY, TEST_SAMPLES, TEST_CANDS = [], [], []
def run_test(tid, desc, expected, samples, checker):
    # samples: list of (pi_a, pi_b, meta)
    if not samples:
        TEST_SUMMARY.append({"test_id": tid, "description": desc, "expected": expected, "n_samples": 0, "n_pass": 0, "status": "NOT_FOUND",
                             "note": "데이터에서 해당 유형 사례를 찾지 못함(만들지 않음)"}); return
    npass = 0
    for k, (a, b, meta) in enumerate(samples, 1):
        res = screen_pair(SEQ(a), SEQ(b)); ok, obs = checker(res, meta); npass += int(ok)
        if k <= 3:
            TEST_SAMPLES.append({"test_id": tid, "sample_no": k, "item_seq_a": SEQ(a), "item_name_a": P_ROW[a].item_name, "item_seq_b": SEQ(b),
                                 "item_name_b": P_ROW[b].item_name, "meta": json.dumps(meta, ensure_ascii=False, default=J), "result": "PASS" if ok else "FAIL", "observed": obs})
            for c in res["candidates"]: TEST_CANDS.append({"test_id": tid, "sample_no": k, **c})
            for h in res["hints"]: TEST_CANDS.append({"test_id": tid, "sample_no": k, **{"pair_class": "FORM_VARIANT_HINT", **h}})
    TEST_SUMMARY.append({"test_id": tid, "description": desc, "expected": expected, "n_samples": len(samples), "n_pass": npass,
                         "status": "PASS" if npass == len(samples) else "FAIL", "note": ""})
def pick(lst, k=N_BATCH): return rng.sample(lst, min(k, len(lst)))

# T1
s1 = []
for c, pis in BY_CODE.items():
    if c in CODE_MULTI or not (2 <= len(pis) <= 20): continue
    ss = [p for p in pis if p in set(sing.pi)]
    if len(ss) >= 2: s1.append((c, ss))
s1 = [(*rng.sample(ss, 2), {"m_code": c}) for c, ss in pick(s1)]
run_test("T1", "동일 M코드 + 동일 성분명 (단일제 2개)", "후보 1건 SAME_MCODE_SAME_NAME", s1,
         lambda r, m: (len(r["candidates"]) == 1 and r["candidates"][0]["pair_class"] == "SAME_MCODE_SAME_NAME" and r["candidates"][0]["a_m_code"] == m["m_code"], "%d건 %s" % (len(r["candidates"]), [c["pair_class"] for c in r["candidates"]])))
# T2
s2 = []
for n in NAME_MULTI:
    cs = [c for c in name_codes[n] if SING_BY_UNIT.get((c, n))]
    if len(cs) >= 2:
        c1, c2 = rng.sample(cs, 2); s2.append((rng.choice(SING_BY_UNIT[(c1, n)]), rng.choice(SING_BY_UNIT[(c2, n)]), {"name": n, "codes": [c1, c2]}))
s2 = pick(s2)
run_test("T2", "동일 성분명 + 서로 다른 M코드 (단일제 2개)", "후보 1건 SAME_NAME_DIFF_MCODE (A 아님)", s2,
         lambda r, m: (len(r["candidates"]) == 1 and r["candidates"][0]["pair_class"] == "SAME_NAME_DIFF_MCODE" and not r["candidates"][0]["type_A_same_mcode"] and r["candidates"][0]["type_C_same_name_diff_mcode"], "%d건 %s" % (len(r["candidates"]), [c["pair_class"] for c in r["candidates"]])))
# T3
s3 = []
scr_multi = set(P.pi[P.screenable & (P.n_ing >= 2)]); sing_set = set(sing.pi)
for c, pis in BY_CODE.items():
    if c in CODE_MULTI or len(pis) > 200: continue
    a = [p for p in pis if p in sing_set]; b = [p for p in pis if p in scr_multi]
    if a and b: s3.append((rng.choice(a), rng.choice(b), {"m_code": c}))
s3 = pick(s3)
run_test("T3", "복합제 vs 단일제 성분 중복", "공유 성분 1건만 후보(나머지 성분은 후보 아님)", s3,
         lambda r, m: (len(r["candidates"]) == 1 and r["candidates"][0]["a_m_code"] == m["m_code"], "%d건 %s" % (len(r["candidates"]), [c["a_m_code"] for c in r["candidates"]])))
# T4
LENS_NAMES = defaultdict(set)
for n in set(UNIT.name_norm): LENS_NAMES[lens(n)].add(n)
s4 = []
for lk, names in LENS_NAMES.items():
    if not lk or len(names) < 2: continue
    names = sorted(names)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            n1, n2 = names[i], names[j]
            u1 = [k for k in SING_BY_UNIT if k[1] == n1]; u2 = [k for k in SING_BY_UNIT if k[1] == n2]
            if u1 and u2:
                base_vs_form = int(lens(n1) == re.sub(r"\s+", "", n1).casefold() or lens(n2) == re.sub(r"\s+", "", n2).casefold())
                s4.append((base_vs_form, rng.choice(SING_BY_UNIT[u1[0]]), rng.choice(SING_BY_UNIT[u2[0]]), {"name_a": n1, "name_b": n2}))
s4.sort(key=lambda x: -x[0]); top = [x for x in s4 if x[0] == 1]; rest = [x for x in s4 if x[0] == 0]
s4 = [(a, b, m) for _, a, b, m in (rng.sample(top, min(N_BATCH, len(top))) + rng.sample(rest, min(max(0, N_BATCH - len(top)), len(rest))))]
run_test("T4", "성분 형태가 다른 사례 (예: X vs X염산염)", "후보 0건 + 형태 힌트 1건 이상 (형태를 제거해 합치지 않음)", s4,
         lambda r, m: (len(r["candidates"]) == 0 and len(r["hints"]) >= 1, "후보 %d / 힌트 %d" % (len(r["candidates"]), len(r["hints"]))))
# T5
pool = list(sing.pi); s5 = []; tries = 0
while len(s5) < N_BATCH and tries < 5000 and len(pool) >= 2:
    tries += 1; a, b = rng.sample(pool, 2); ra, rb = ELIG_BY_PI[a][0], ELIG_BY_PI[b][0]
    if ra.m_code != rb.m_code and ra.name_norm != rb.name_norm and lens(ra.name_norm) != lens(rb.name_norm): s5.append((a, b, {}))
run_test("T5", "중복 성분 없는 사례 (무작위 단일제 쌍)", "후보 0건, 힌트 0건", s5,
         lambda r, m: (len(r["candidates"]) == 0 and len(r["hints"]) == 0 and r["status"] == "OK", "후보 %d / 힌트 %d / %s" % (len(r["candidates"]), len(r["hints"]), r["status"])))
# T6a / T6b
uns = list(P.pi[P.unscreenable]); part = list(P.pi[P.partial]); okp = list(sing.pi)
s6 = [(a, rng.choice(okp), {"kind": "no_ingredient_info"}) for a in pick(uns)] if okp else []
run_test("T6a", "성분정보 누락/불명확로 후보 계산 불가 제품", "UNSCREENABLE, 후보 0건", s6,
         lambda r, m: (r["status"].startswith("UNSCREENABLE") and len(r["candidates"]) == 0, r["status"] + " / 후보 %d" % len(r["candidates"])))
def chk6b(r, m):
    bad = {(c, nm) for c, nm, _ in (r["unclear_a"] + r["unclear_b"])}
    leaked = [c for c in r["candidates"] if (c["a_m_code"], c["a_name_raw"]) in bad or (c["b_m_code"], c["b_name_raw"]) in bad]
    return (len(leaked) == 0 and (len(r["unclear_a"]) + len(r["unclear_b"])) > 0, "D 행 %d개 / D 행이 후보에 포함된 건수 %d" % (len(r["unclear_a"]) + len(r["unclear_b"]), len(leaked)))
s6b = [(a, rng.choice(okp), {"kind": "partially_unclear"}) for a in pick(part)] if okp else []
run_test("T6b", "일부 성분만 불명확한 제품 (D 행 제외 확인)", "D 행이 후보에 나타나지 않음", s6b, chk6b)

TS_DF = pd.DataFrame(TEST_SUMMARY); show(TS_DF[["test_id", "description", "n_samples", "n_pass", "status"]], 20)
# ---- 전역 불변 조건 ----
INV = []
allc = pd.DataFrame([c for c in TEST_CANDS if c.get("candidate")])
INV.append({"invariant": "모든 후보의 검증 상태가 UNVERIFIED_CANDIDATE", "ok": bool((allc.verification_status == CAND_STATUS).all()) if len(allc) else True})
INV.append({"invariant": "후보는 동일 코드 또는 동일 정규화명을 가짐", "ok": bool(((allc.a_m_code == allc.b_m_code) | (allc.a_name_norm == allc.b_name_norm)).all()) if len(allc) else True})
INV.append({"invariant": "후보의 정규화명 == norm(기존 성분명)", "ok": bool(all(norm(a) == n for a, n in zip(allc.a_name_raw, allc.a_name_norm))) if len(allc) else True})
INV.append({"invariant": "자기 자신과의 쌍 없음", "ok": bool((allc.a_item_seq != allc.b_item_seq).all()) if len(allc) else True})
INV.append({"invariant": "형태 힌트는 후보 목록(candidate=True)에 없음", "ok": all(not (c.get("pair_class") == "FORM_VARIANT_HINT" and c.get("candidate")) for c in TEST_CANDS)})
sym_ok = True
for t in TEST_SAMPLES[:30]:
    r1 = screen_pair(t["item_seq_a"], t["item_seq_b"]); r2 = screen_pair(t["item_seq_b"], t["item_seq_a"])
    sym_ok &= len(r1["candidates"]) == len(r2["candidates"]) and len(r1["hints"]) == len(r2["hints"])
INV.append({"invariant": "A,B 순서를 바꿔도 결과 건수 동일(대칭)", "ok": bool(sym_ok)})
INVD = pd.DataFrame(INV); show(INVD, 10)
st = set(TS_DF.status)
record("CELL 5", "FAIL" if ("FAIL" in st or not INVD.ok.all()) else ("WARN" if "NOT_FOUND" in st else "PASS"),
       ("테스트 실패 또는 불변 조건 위반" if ("FAIL" in st or not INVD.ok.all()) else ("데이터에 없어 NOT_FOUND인 유형: " + ",".join(TS_DF[TS_DF.status == "NOT_FOUND"].test_id)) if "NOT_FOUND" in st else ""))
if "FAIL" in st or not INVD.ok.all(): print("※ FAIL 상세는 TEST_SAMPLES 참조"); show(pd.DataFrame(TEST_SAMPLES).query("result == 'FAIL'"), 10)

## CELL 6 · 전체 데이터 범위 지표 (분자·분모 명시)
쌍을 전수 생성하지 않고 인덱스로 계산합니다. **후보가 많다는 사실은 정확도의 증거가 아닙니다.** 이 표는 "탐지 가능 범위"만 말합니다. 검증 표본 결과를 전체의 정확도로 일반화하지 않습니다.

In [ ]:
if RUN.get("CELL 5", {}).get("status") == "FAIL": raise RuntimeError("CELL 5가 FAIL — 먼저 해결")
need("CELL 5")
M = []
def met(mid, label, num, den, den_def, caveat=""):
    M.append({"id": mid, "label": label, "numerator": int(num), "denominator": int(den), "denominator_def": den_def, "pct": pct(int(num), int(den)), "caveat": caveat})
n_prod = len(P); n_ing_rows = len(ING)
met("prod_screenable", "후보 계산 가능 제품(모든 성분 행이 명확)", P.screenable.sum(), n_prod, "전체 제품")
met("prod_partial", "일부 성분 행이 불명확(D)인 제품", P.partial.sum(), n_prod, "전체 제품")
met("prod_unscreenable", "후보 계산 불가 제품(성분 정보 없음/전부 불명확)", P.unscreenable.sum(), n_prod, "전체 제품")
met("row_D", "D(불명확) 성분 행", ING.unclear_D.sum(), n_ing_rows, "전체 성분 행")
def has_partner(pi, hf_ok=True):
    for r in ELIG_BY_PI.get(pi, []):
        for n in (len(BY_CODE[r.m_code]), len(BY_NAME[r.name_norm])):
            if n > 1 and (hf_ok or n <= HF_THRESHOLD): return True
    return False
elig_prods = [pi for pi in P.pi if ELIG_BY_PI.get(pi)]
met("prod_with_partner", "다른 제품과 동일 M코드/성분명을 공유하는 제품(후보 파트너 존재)", sum(has_partner(p) for p in elig_prods), len(elig_prods), "후보 계산 가능 성분이 1개 이상인 제품", "공유 사실일 뿐 동일성 확인 아님")
met("prod_with_partner_nonHF", "위 중 고빈도 단위(>%d제품) 공유를 제외해도 파트너 존재" % HF_THRESHOLD, sum(has_partner(p, False) for p in elig_prods), len(elig_prods), "후보 계산 가능 성분이 1개 이상인 제품")
cprod = set(); cunits = set(zip(UNIT[UNIT.type_C_unit].m_code, UNIT[UNIT.type_C_unit].name_norm))
for r in ELIG.itertuples():
    if (r.m_code, r.name_norm) in cunits: cprod.add(r.pi)
met("prod_with_C_unit", "'같은 이름·다른 M코드(C)' 단위를 가진 제품", len(cprod), len(elig_prods), "후보 계산 가능 성분이 1개 이상인 제품", "C 단위 성분은 이름만으로 동일성을 판단할 수 없음")
met("unit_1to1", "코드↔이름이 서로 1:1인 성분 단위", ((UNIT.code_n_names == 1) & (UNIT.name_n_codes == 1)).sum(), len(UNIT), "성분 단위(M코드+정규화명)", "프로젝트 저장본 안에서의 1:1일 뿐")
met("unit_C", "이름이 M코드 2개 이상에 걸친 단위(C)", UNIT.type_C_unit.sum(), len(UNIT), "성분 단위")
met("unit_HF", "고빈도 성분 단위(>%d제품)" % HF_THRESHOLD, UNIT.high_frequency.sum(), len(UNIT), "성분 단위")
met("prod_count_mismatch", "ITEM_INGR_CNT와 고유 M코드 수가 다른 제품", P.count_mismatch.sum(), int(((P.cnt.notna()) & (P.n_codes > 0)).sum()), "선언 수와 M코드가 모두 있는 제품", "경고 지표(후보 제외 사유는 아님)")
met("prod_within_repeat", "한 제품 안에서 같은 M코드가 반복되는 제품", P.within_product_repeat.sum(), int((P.n_codes > 0).sum()), "M코드가 있는 제품", "합치지 않고 그대로 유지")
METRICS = pd.DataFrame(M); show(METRICS[["id", "numerator", "denominator", "denominator_def", "pct"]], 20)
code_sizes = pd.Series({c: len(s) for c, s in BY_CODE.items()}); name_sizes = pd.Series({n: len(s) for n, s in BY_NAME.items()})
EST = {"pairs_sharing_code_est": int((code_sizes * (code_sizes - 1) // 2).sum()), "pairs_sharing_name_est": int((name_sizes * (name_sizes - 1) // 2).sum()),
       "note": "전수 생성하지 않음. 고빈도 단위가 대부분을 차지하며 두 값은 중복 포함."}
TOP_HF = UNIT.sort_values("n_products", ascending=False).head(15)[["m_code", "name_norm", "n_products"]]
print("추정 후보 쌍 수(전수 생성 안 함):", EST); print("\n[고빈도 단위 상위 15]"); show(TOP_HF, 15)
record("CELL 6", "PASS")

## CELL 7 · 제한적 Ingredient Screening 확장 가능성 평가
평가는 **코드 규칙**으로 계산합니다(임계값은 설계 파라미터이며 의학적 기준이 아님). **어떤 경우에도 자동 경보·자동 Screening 연결은 허용하지 않습니다.** 공식 동일성 검증 NO-GO를 그대로 유지합니다.

In [ ]:
need("CELL 6")
mt = METRICS.set_index("id")
tests_ok = bool(set(TS_DF.status) <= {"PASS", "NOT_FOUND"}) and bool(INVD.ok.all())
not_found = list(TS_DF[TS_DF.status == "NOT_FOUND"].test_id)
VERDICT = "NOT_READY" if not tests_ok else "LIMITED_PILOT_CANDIDATE_DISPLAY_ONLY"
ALLOWED = ["제품 쌍/복약 목록에서 '성분 중복 *후보*' 표시 (사람 검토용, 설명 문구에 '후보·미검증' 명시)",
           "후보 근거(동일 M코드/동일 정규화명/이름 동일·코드 상이)와 원천 위치(ITEM_SEQ, 컬럼, 수집 페이지) 함께 표시",
           "D(불명확) 성분과 후보 계산 불가 제품은 '판단 불가'로 명시",
           "고빈도 성분 단위는 접어서 표시하거나 별도 안내(제외 아님)"]
FORBIDDEN = ["자동 경보 / DUR 판정과 결합 / 위험도 점수화", "염·수화물·에스터·입체 접두어를 제거해 합친 중복 판정", "C 유형(같은 이름·다른 M코드)을 '동일 성분'으로 단정",
             "용량 환산 또는 용량 동등성 판단", "기존 Screening Engine·RAG·LLM·UI에 자동 연결"]
EVAL = {"verdict": VERDICT, "official_identity_validation_status": OFFICIAL_IDENTITY_STATUS, "tests_ok": tests_ok, "tests_not_found": not_found,
        "allowed_scope": ALLOWED, "forbidden": FORBIDDEN,
        "basis": "프로젝트 저장본 기준 측정. 공식 근거 없음. 테스트 통과는 '규칙대로 동작함'을 뜻하며 '성분 동일성이 맞음'을 뜻하지 않음."}
print("판정:", VERDICT, "| 공식 성분 동일성 검증:", OFFICIAL_IDENTITY_STATUS)
print("\n[허용 범위]"); [print(" -", x) for x in ALLOWED]
print("\n[금지]"); [print(" -", x) for x in FORBIDDEN]
if not_found: print("\n※ 데이터에 없어 검증하지 못한 테스트:", not_found)
record("CELL 7", "WARN" if (not tests_ok or not_found) else "PASS", ("테스트 미통과" if not tests_ok else "") + (" NOT_FOUND: " + ",".join(not_found) if not_found else ""))

## CELL 8 · 저장 · 보고서 · 불변 확인
새 실행 폴더에만 저장합니다. 입력 parquet과 기존 보고서 폴더의 SHA256이 시작 시와 같은지 확인합니다. **FAIL = 입력 또는 기존 파일 변경 감지.**

In [ ]:
need("CELL 7")
ING_OUT = ING[["item_seq", "item_name", "pos", "m_code", "name_raw", "name_norm", "form_markers", "d_reasons", "unclear_D"]].copy()
ING_OUT.to_csv(OUT / "01_ingredient_rows.csv", index=False, encoding="utf-8-sig")
UNIT.to_csv(OUT / "02_ingredient_unit_index.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(TEST_CANDS).to_csv(OUT / "03_test_candidate_pairs.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(TEST_SAMPLES).to_csv(OUT / "04_test_samples.csv", index=False, encoding="utf-8-sig")
TS_DF.to_csv(OUT / "04_test_results.csv", index=False, encoding="utf-8-sig")
P[["item_seq", "item_name", "n_ing", "n_elig", "screenable", "partial", "unscreenable", "within_product_repeat", "count_mismatch"]].to_csv(OUT / "05_product_screenability.csv", index=False, encoding="utf-8-sig")
META = {"run_folder": str(OUT), "input": str(SRC), "input_sha256": SRC_SHA, "seed": SEED, "hf_threshold": HF_THRESHOLD, "n_batch": N_BATCH, "official_identity_validation_status": OFFICIAL_IDENTITY_STATUS,
        "metrics": METRICS.to_dict("records"), "pair_estimates": EST, "tests": TS_DF.to_dict("records"), "invariants": INVD.to_dict("records"), "evaluation": EVAL,
        "cell_status_so_far": {k: v["status"] for k, v in RUN.items()}}
(OUT / "06_metrics_and_evaluation.json").write_text(json.dumps(META, ensure_ascii=False, indent=2, default=J), encoding="utf-8")
def mrow(i): r = mt.loc[i]; return "| %s | %d | %d | %s | %s |" % (r.label, r.numerator, r.denominator, r.denominator_def, "%.2f%%" % r.pct if r.pct is not None else "N/A")
rep = ["# 15 성분 중복 후보 탐지 프로토타입 보고서 (프로젝트 저장본 기준 · 비자동)", "",
       "- 실행 폴더: `%s`" % OUT, "- 입력: `%s` / SHA256 `%s`" % (SRC, SRC_SHA), "- 시드 %d / 고빈도 임계 %d / 테스트 표본 수 ≤ %d" % (SEED, HF_THRESHOLD, N_BATCH),
       "- **공식 성분 동일성 검증 판정: %s (유지)**. 이 보고서의 모든 후보는 `%s`." % (OFFICIAL_IDENTITY_STATUS, CAND_STATUS), "",
       "## 1. 규칙", "- 정규화: NFKC + zero-width 제거 + 공백 정리만. 염·수화물·에스터·입체 접두어·괄호·함량 보존. 대소문자 미병합.",
       "- 같은 M코드/같은 이름은 후보일 뿐이며 약학적 동일성·임상 위험을 확정하지 않음.", "",
       "## 2. 범위 지표 (분자·분모)", "| 지표 | 분자 | 분모 | 분모 정의 | % |", "|---|---|---|---|---|",
       *[mrow(i) for i in METRICS.id], "", "추정 후보 쌍 수(전수 생성 안 함): %s" % json.dumps(EST, ensure_ascii=False), "",
       "## 3. 테스트 결과", "| 테스트 | 설명 | 표본 | 통과 | 상태 |", "|---|---|---|---|---|",
       *["| %s | %s | %d | %d | %s |" % (r.test_id, r.description, r.n_samples, r.n_pass, r.status) for r in TS_DF.itertuples()], "",
       "- 불변 조건: " + "; ".join("%s=%s" % (r.invariant, "OK" if r.ok else "FAIL") for r in INVD.itertuples()),
       "- 테스트 통과는 '규칙대로 동작함'이며 '성분 동일성이 맞음'을 의미하지 않음. 표본 결과를 전체 정확도로 일반화하지 않음.", "",
       "## 4. 제한적 Ingredient Screening 평가", "**%s**" % VERDICT, "", "허용: " + " / ".join(ALLOWED), "", "금지: " + " / ".join(FORBIDDEN), "",
       "## 5. 셀 상태", "| 셀 | 상태 | 비고 |", "|---|---|---|", *["| %s | %s | %s |" % (k, v["status"], v["detail"]) for k, v in RUN.items()], "",
       "> 이 보고서는 사용자의 Colab 실제 실행 결과에서만 유효합니다."]
(OUT / "07_candidate_prototype_report.md").write_text("\n".join(rep), encoding="utf-8")
# ---- 불변 확인 ----
sha_after = sha256_file(SRC); cur = snap(WATCH_DIRS)
changed = snap_changed(SNAP0, cur); added = [p for p in cur if p not in SNAP0]
inv = {"input_unchanged": sha_after == SRC_SHA, "previous_reports_unchanged": not changed, "files_added_in_previous_report_dirs": added[:10]}
(OUT / "00_integrity.json").write_text(json.dumps({"input_sha256_before": SRC_SHA, "input_sha256_after": sha_after, **inv, "watched_files": len(SNAP0), "snapshot_modes": dict(SNAP_MODES), "snapshot_issues": SNAP_ISSUES}, ensure_ascii=False, indent=2), encoding="utf-8")
print("불변 확인:", inv)
for f in sorted(OUT.iterdir()): print(" ", f.name, f.stat().st_size, "bytes")
if not inv["input_unchanged"] or not inv["previous_reports_unchanged"] or added:
    record("CELL 8", "FAIL", "입력/기존 보고서 변경 감지: %s" % inv); raise RuntimeError("불변 확인 실패")
record("CELL 8", "WARN" if any(v["status"] == "WARN" for v in RUN.values()) else "PASS", "판정 %s — 공식 동일성 NO-GO 유지" % VERDICT)
print("\n※ 이 결과는 프로젝트 저장본 기준 후보 탐지입니다. 성분 동일성은 검증되지 않았습니다.")